# Synthetic GelMA Hydrogel Subunit Protein Design (Waterloo iGEM 2025)

***dependencies***

In [42]:
import numpy as np
import glob
import os
import pandas as pd
from Bio import SeqIO
import re
import sys
sys.path.append('scripts/')
from AF2_HelperFunctions import *
import tqdm

-------------------------

### Generate Backbone Repeating GPH Motifs

sample `num_repeats` 6-10

In [ ]:
# sydney put code here

--------------------

### Block for MPNN Stuff - @ curtis

-----------

### Analysis of MPNN Outputs

#### 1. Load in .fa files and generate AF Jobs. Store seqs to frame `subunit_seqs' w/ relevant data

In [6]:
mpnn_outputs = glob.glob("output/mpnn/*.fa")
mpnn_outputs

['output/mpnn/fold_collagen_6_repeats_model_0.fa']

In [18]:
def parse_mpnn_fastas(fasta_dir):
    records = []

    for fname in os.listdir(fasta_dir):
        if fname.endswith('.fa') or fname.endswith('.fasta'):
            model_name = os.path.splitext(fname)[0]
            path = os.path.join(fasta_dir, fname)
            for record in SeqIO.parse(path, "fasta"):
                meta = {
                    "model": model_name,
                    "sequence": str(record.seq),
                    "header": record.description
                }

                # Extract metadata from the FASTA header using regex
                meta["temperature"] = float(re.search(r'T=([\d.]+)', record.description).group(1)) if "T=" in record.description else None
                meta["sample"] = int(re.search(r'sample=(\d+)', record.description).group(1)) if "sample=" in record.description else None
                meta["score"] = float(re.search(r'score=([\d.]+)', record.description).group(1)) if "score=" in record.description else None
                meta["seq_recovery"] = float(re.search(r'seq_recovery=([\d.]+)', record.description).group(1)) if "seq_recovery=" in record.description else None

                records.append(meta)

    return pd.DataFrame(records)


In [22]:
subunit_seqs = parse_mpnn_fastas("output/mpnn")
subunit_seqs.head(2)

,model,sequence,header,temperature,sample,score,seq_recovery
0,fold_collagen_6_repeats_model_0,GPHGPHGPHGPHGPHGPH,"fold_collagen_6_repeats_model_0, score=1.5326,...",NaN,NaN,1.5326,NaN
1,fold_collagen_6_repeats_model_0,GETGLYGKYGLYGPYGPY,"T=0.15, sample=1, score=1.0761, global_score=1...",0.15,1.0,1.0761,0.4444


In [23]:
def WriteSuperfoldCommand(fa, outdir, ref=None, num_recycles=5, model=4, initial_guess=False):
    if (initial_guess and ref == None):
        raise ValueError('Need to provide reference pdb if intial_guess')
    cmd= f'''apptainer run --nv --bind /net:/net --bind /scratch:/scratch /net/software/containers/superfold.sif /net/software/superfold/run_superfold.py {fa} \
--models {model} \
--out_dir {outdir} \
--max_recycles {num_recycles} '''
    if ref != None:
        cmd += f'--reference {ref} '
    if initial_guess:
        cmd += f'--initial_guess {ref} '
    return cmd

In [27]:
import os

af2_folder = 'output/af2/'
os.makedirs(af2_folder, exist_ok=True)

# Initialize counters and batch parameters
design_ctr = 0
batch_ctr = 0
batch_size = 100

af2_cmds = []

# Loop through sequences and write AF2 input files
for i in range(len(subunit_seqs)):
    seq = subunit_seqs.loc[i, 'sequence']
    model_id = subunit_seqs.loc[i, 'model']
    header = subunit_seqs.loc[i, 'header']
    model_name = f"{header}_{model_id}"

    model_dir = os.path.join(af2_folder, model_id)
    os.makedirs(model_dir, exist_ok=True)

    # Write .fa file
    fasta_path = os.path.join(model_dir, f"{model_name}.fa")
    with open(fasta_path, 'w') as f:
        f.write(f">{model_name}\n{seq}\n")

    af2_cmd = WriteSuperfoldCommand(fasta_path, model_dir)
    af2_cmds.append(af2_cmd)

    design_ctr += 1

# Write command list to task file
os.makedirs("tasks", exist_ok=True)
with open("tasks/af2_monomer.tasks", 'w') as f:
    for cmd in af2_cmds:
        f.write(cmd + '\n')

print(f'{design_ctr} AF2 tasks written')
print(f'{batch_ctr} batch files created')

21 AF2 tasks written
0 batch files created


submit `af2_monomer.tasks` to slurm, or copy paste to terminal

In [31]:
import os
import re

# Utility to sanitize strings for safe filenames
def sanitize(name):
    return re.sub(r'[^a-zA-Z0-9_]', '_', name)

# Set output base folder
af2_folder = "output/af2/"
os.makedirs(af2_folder, exist_ok=True)

af2_cmds = []

for i in range(len(subunit_seqs)):
    seq = subunit_seqs.loc[i, 'sequence']
    header = subunit_seqs.loc[i, 'header']
    model_id = subunit_seqs.loc[i, 'model']

    # Use full header + model as original descriptive name
    raw_model_name = f"{header}_{model_id}"
    # Safe version for file naming
    sanitized_model_name = sanitize(raw_model_name)

    model_dir = os.path.join(af2_folder, model_id)
    os.makedirs(model_dir, exist_ok=True)

    fasta_path = os.path.join(model_dir, f"{sanitized_model_name}.fa")
    with open(fasta_path, 'w') as f:
        f.write(f">{raw_model_name}\n{seq}\n")

    cmd = (
        f"apptainer run --nv --bind /net:/net --bind /scratch:/scratch "
        f"/net/software/containers/superfold.sif /net/software/superfold/run_superfold.py "
        f"{fasta_path} --models 4 --out_dir {model_dir} --max_recycles 5"
    )
    af2_cmds.append(cmd)

# Write command list to task file
os.makedirs("tasks", exist_ok=True)
with open("tasks/af2_monomer.tasks", 'w') as f:
    for cmd in af2_cmds:
        f.write(cmd + '\n')

print(f'{design_ctr} AF2 tasks written')
print(f'{batch_ctr} batch files created')

21 AF2 tasks written
0 batch files created


#### 2. read in AF2 metrics

In [45]:
predictions = f'output/af2/'
all_preds = glob.glob(f"{predictions}*/*.json")
AF2_df = LoadAF2Metrics(all_preds)
AF2_df.to_csv('AF2_metrics_before_RMSD.csv')

AF2_df.head(2)

100%|██████████| 18/18 [00:00<00:00, 8064.25it/s]


,mean_plddt,recycles,tol,model,type,seed,mean_pae_interaction,mean_pae_intra_chain_A,mean_pae_intra_chain,mean_pae,pTMscore,elapsed_time,filepath,design,PDB
0,62.664131,5,0.538243,4,monomer_ptm,0,NaN,10.451608,10.451608,10.917353,0.022048,56.752126,output/af2/fold_collagen_6_repeats_model_0/T=0...,"T=0.15, sample=5, score=1.2371, global_score=1...",output/af2/fold_collagen_6_repeats_model_0/T=0...
1,55.295860,5,0.719819,4,monomer_ptm,0,NaN,11.445728,11.445728,11.961148,0.022001,56.159701,output/af2/fold_collagen_6_repeats_model_0/T=0...,"T=0.15, sample=9, score=1.1214, global_score=1...",output/af2/fold_collagen_6_repeats_model_0/T=0...


In [46]:

# Initialize the new columns with NaN values if they don't exist
if 'rmsd_to_reference' not in AF2_df.columns:
    AF2_df['rmsd_to_reference'] = float('nan')


ctr = 0 
for i, r in tqdm.tqdm(AF2_df.iterrows(), total=AF2_df.shape[0]):
    if pd.notna(r['rmsd_to_reference']):
        continue  # Skip rows where RMSD has already been calculated

    # hard-code in the parent_pdb for now
    ref_pdb = glob.glob('input/parent/*.pdb')[0]
    rmsd = rmsd_kabsch(ref_pdb, r['PDB'], ['A']) # calculate ca-rmsd over chain A
    try:
        rmsd = rmsd_kabsch(ref_pdb, r['PDB'], ['A']) # calculate ca-rmsd over chain A
    except Exception as e:
        ctr += 1
    else:
        AF2_df.at[i, 'rmsd_to_reference'] = rmsd

print(f'{ctr} designs ({round(ctr/len(AF2_df))}%) had a problem in the alignment.')

# Drop rows with NaN values in the 'rmsd_to_reference' column
# AF2_df.dropna(subset=['rmsd_to_reference'], inplace=True)

# Save the updated DataFrame
AF2_df.to_csv('AF2_monomer_metrics.csv')

AF2_df

100%|██████████| 18/18 [00:00<00:00, 144.25it/s]

0 designs (0%) had a problem in the alignment.


,mean_plddt,recycles,tol,model,type,seed,mean_pae_interaction,mean_pae_intra_chain_A,mean_pae_intra_chain,mean_pae,pTMscore,elapsed_time,filepath,design,PDB,rmsd_to_reference
0,62.664131,5,0.538243,4,monomer_ptm,0,NaN,10.451608,10.451608,10.917353,0.022048,56.752126,output/af2/fold_collagen_6_repeats_model_0/T=0...,"T=0.15, sample=5, score=1.2371, global_score=1...",output/af2/fold_collagen_6_repeats_model_0/T=0...,8.007898
1,55.295860,5,0.719819,4,monomer_ptm,0,NaN,11.445728,11.445728,11.961148,0.022001,56.159701,output/af2/fold_collagen_6_repeats_model_0/T=0...,"T=0.15, sample=9, score=1.1214, global_score=1...",output/af2/fold_collagen_6_repeats_model_0/T=0...,7.456590
2,55.281086,5,0.478376,4,monomer_ptm,0,NaN,10.979735,10.979735,11.518855,0.022317,56.452943,output/af2/fold_collagen_6_repeats_model_0/T=0...,"T=0.15, sample=2, score=1.1168, global_score=1...",output/af2/fold_collagen_6_repeats_model_0/T=0...,7.588369
3,61.943169,5,0.187067,4,monomer_ptm,0,NaN,10.415311,10.415311,10.871015,0.022230,56.389590,output/af2/fold_collagen_6_repeats_model_0/T=0...,"T=0.15, sample=18, score=1.0587, global_score=...",output/af2/fold_collagen_6_repeats_model_0/T=0...,6.878652
4,59.645245,5,0.342754,4,monomer_ptm,0,NaN,10.421756,10.421756,10.937842,0.022519,56.799219,output/af2/fold_collagen_6_repeats_model_0/T=0...,"T=0.15, sample=6, score=1.0945, global_score=1...",output/af2/fold_collagen_6_repeats_model_0/T=0...,6.419917
5,63.107380,5,0.261730,4,monomer_ptm,0,NaN,8.986873,8.986873,9.306001,0.026191,55.956638,output/af2/fold_collagen_6_repeats_model_0/T=0...,"T=0.15, sample=12, score=1.1800, global_score=...",output/af2/fold_collagen_6_repeats_model_0/T=0...,5.869031
6,58.745472,5,0.078573,4,monomer_ptm,0,NaN,11.028003,11.028003,11.460927,0.021694,56.161637,output/af2/fold_collagen_6_repeats_model_0/T=0...,"T=0.15, sample=10, score=1.0961, global_score=...",output/af2/fold_collagen_6_repeats_model_0/T=0...,6.910998
7,60.602146,5,0.126465,4,monomer_ptm,0,NaN,9.976798,9.976798,10.468680,0.023467,56.818839,output/af2/fold_collagen_6_repeats_model_0/T=0...,"T=0.15, sample=1, score=1.0761, global_score=1...",output/af2/fold_collagen_6_repeats_model_0/T=0...,6.905877
8,62.537251,5,0.116559,4,monomer_ptm,0,NaN,10.803412,10.803412,11.275718,0.022597,56.391663,output/af2/fold_collagen_6_repeats_model_0/T=0...,"T=0.15, sample=20, score=1.1606, global_score=...",output/af2/fold_collagen_6_repeats_model_0/T=0...,7.639337
9,56.502766,5,0.400224,4,monomer_ptm,0,NaN,11.632370,11.632370,12.116806,0.021772,56.653545,output/af2/fold_collagen_6_repeats_model_0/T=0...,"T=0.15, sample=7, score=1.0713, global_score=1...",output/af2/fold_collagen_6_repeats_model_0/T=0...,7.103045
